# Crop annotation

Pulls one crop at a time from the existing WS-DINO loader, shows the **t1**
global crop as a max-intensity projection, and lets you assign a single class
with a button click. Each label writes the full 3D t1 crop (`.npy`), a MIP
preview (`.png`), and an append-only JSONL record under `OUT_DIR` (see
`data/annotate_io.py`).

Augmentations are **disabled** here so you label clean crops.

**One dataset at a time.** To build an even spread across datasets and classes,
you annotate a single dataset per session (set `DATASET_INDEX`). A live
**dataset × class** table shows per-dataset row totals and per-class column
totals (across all datasets) so you can keep the counts balanced.

**Resume / dedup.** A fresh random seed each session, so resume shows new crops.
Crops already **labeled** or **skipped** are suppressed. **Skip** stashes a crop
to revisit in bulk later (the *Review skipped* section).

Run all cells, then click a class button per crop. Use the **predicty** kernel.

In [ ]:
%matplotlib inline
import sys, os, random
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import ipywidgets as widgets
from IPython.display import display, clear_output

from configs.config import Config
from data.dataset import ZarrCellDataset, MultiZarrDataset
from data.annotate_io import (
    make_uid, save_annotation, save_skip, relabel_skip,
    load_seen, load_skipped, load_suppressed, load_pending_skips,
    load_counts, pending_skip_counts,
    load_labeled_records, load_heldout, load_heldout_records,
    export_heldout, remove_annotations,
)


# --- display helpers (mirrored from notebooks/visualize.ipynb) ---
def mip(volume):
    """max projection across z; bright structures stand out."""
    return volume.max(axis=0)


def percentile_limits(values, low=1.0, high=99.0):
    """display limits robust to a few bright outliers."""
    lo, hi = np.percentile(np.asarray(values), [low, high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.min(values)), float(np.max(values))
    if hi <= lo:
        hi = lo + 1e-6
    return float(lo), float(hi)


# --- live counts table ---
def _ds_short(path):
    """short, mostly-unique label for a zarr path (idx prefix disambiguates)."""
    parts = path.rstrip('/').split('/')
    name = '/'.join(parts[-3:-1]) if len(parts) >= 3 else parts[-1]
    return name if len(name) <= 40 else '…' + name[-39:]


def _ds_long(path):
    """longer label for a zarr path, with more context but still truncated."""
    parts = path.rstrip('/').split('/')
    name = '/'.join(parts[-4:-1]) if len(parts) >= 4 else parts[-1]
    return name if len(name) <= 60 else '…' + name[-59:]


def counts_dataframe(out_dir, master_paths, classes):
    """dataset x class table with row totals, a 'skip' column, and an 'all' row
    of per-class column totals + grand total. Re-read from disk each call."""
    lab = load_counts(out_dir)
    pend = pending_skip_counts(out_dir)
    cols = list(classes) + ['unsure']
    rows = {}
    for idx, path in enumerate(master_paths):
        per = lab.get(idx, {})
        r = {c: int(per.get(c, 0)) for c in cols}
        r['skip'] = int(pend.get(idx, 0))
        r['total'] = sum(int(per.get(c, 0)) for c in cols)
        rows[f'[{idx}] {_ds_short(path)}'] = r
    df = pd.DataFrame.from_dict(rows, orient='index', columns=cols + ['skip', 'total'])
    if len(df):
        df.loc['all'] = df.sum()
    return df


def render_counts(box, out_dir, master_paths, classes, current_idx):
    df = counts_dataframe(out_dir, master_paths, classes)
    cur = f'[{current_idx}] {_ds_short(master_paths[current_idx])}'

    def _hl(row):
        on = (row.name == cur)
        return ['background-color: #fff3b0' if on else '' for _ in row]

    box.value = df.style.apply(_hl, axis=1).format(precision=0).to_html()

## Config

Edit `CLASSES`, pick a dataset with `DATASET_INDEX` (the printed list is indexed
by `dataset_idx`), and set `LOCAL_MOUNT` if running on the Mac (remaps cluster
`/groups/...` paths to `/Volumes/...`).

`SESSION_SEED = None` draws a fresh random region each session (recommended).

In [2]:
# classes you want to annotate into (single label per crop)
CLASSES = ['empty', 'single_cells', 'clumps', 'streams', 'aggregate', 'slug']

# where labels + crops are written (gitignored)
OUT_DIR = '../annotations/labels'

# None -> fresh random crops each session; set an int for a reproducible order
SESSION_SEED = None

# remap cluster paths to the local mount when running on the Mac
LOCAL_MOUNT = True
def _maybe_remap(p):
    return p.replace('/groups/sgro/sgrolab/', '/Volumes/sgrolab/') if LOCAL_MOUNT else p

# canonical dataset list from the training yaml: index here == dataset_idx.
# CANONICAL_PATHS are stored in records (portable); MASTER_ZARR_PATHS are loaded.
_base = Config.from_yaml('../configs/train.yaml')
CANONICAL_PATHS = list(_base.zarr_paths) if _base.zarr_paths else [_base.zarr_path]
MASTER_ZARR_PATHS = [_maybe_remap(p) for p in CANONICAL_PATHS]
print('available datasets (set DATASET_INDEX):')
for i, p in enumerate(MASTER_ZARR_PATHS):
    print(f'  [{i}] {_ds_short(p)}')

# which dataset to annotate this session
DATASET_INDEX = 19

cfg = Config.from_yaml('../configs/train.yaml')
cfg.zarr_paths = None
cfg.zarr_path = MASTER_ZARR_PATHS[DATASET_INDEX]
cfg.num_workers = 0
cfg.seed = SESSION_SEED if SESSION_SEED is not None else random.randrange(2**31)
for aug in (cfg.aug_intensity, cfg.aug_gamma, cfg.aug_noise,
            cfg.aug_blur, cfg.aug_percrop_norm):
    aug.enabled = False

print()
print(f'annotating [{DATASET_INDEX}] {cfg.zarr_path}')
print('classes:', CLASSES, '| out_dir:', os.path.abspath(OUT_DIR), '| seed:', cfg.seed)

available datasets (set DATASET_INDEX):
  [0] …yolite_mixin_test42_2024-01-12/Field_B2
  [1] …yolite_mixin_test42_2024-01-12/Field_B3
  [2] …yolite_mixin_test41_2024-01-09/Field_B3
  [3] …yolite_mixin_test41_2024-01-09/Field_B2
  [4] …yolite/cryolite_mixin_test43_2024-01-16
  [5] …yolite/cryolite_mixin_test44_2024-01-17
  [6] …yolite/cryolite_mixin_test45_2024-01-30
  [7] …yolite/cryolite_mixin_test48_2024-02-08
  [8] …yolite/cryolite_mixin_test49_2024-02-08
  [9] …yolite/cryolite_mixin_test52_2024-02-21
  [10] …yolite/cryolite_mixin_test53_2025-02-22
  [11] …yolite/cryolite_mixin_test54_2024-02-22
  [12] …yolite/cryolite_mixin_test57_2024-02-29
  [13] …yolite/cryolite_mixin_test60_2024-03-06
  [14] …yolite/cryolite_mixin_test61_2024-03-07
  [15] …yolite/cryolite_mixin_test62_2024-03-08
  [16] …yolite/cryolite_mixin_test64_2024-04-04
  [17] …te2_WS205_overnight_day2_ERH Red FarRed
  [18] …test65_2024-04-16/NC28.1_overnight_day1
  [19] …yolite/cryolite_mixin_test66_2024-04-30

annotatin

## Build the dataset + iterator

In [3]:
ds = ZarrCellDataset(cfg)
it = iter(ds)

# suppress anything already labeled OR skipped (uids are dataset-scoped, so
# this is correct across datasets even though we load one at a time)
suppress = load_suppressed(OUT_DIR)
n_labeled = len(load_seen(OUT_DIR))
n_skipped = len(load_skipped(OUT_DIR))
print(f'resuming: {n_labeled} labeled, {n_skipped} skipped (both suppressed)')

[ZarrCellDataset] opened /Volumes/sgrolab/jennifer/cryolite/cryolite_mixin_test66_2024-04-30/ERH_2024-04-30_mixin66_plate1_day1_ERH Red FarRed.zarr shape=(139, 2, 218, 1906, 1443) chunks=((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (1, 1), (1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

## Annotate

Click a class to save the current crop and advance. **Unsure** saves with the
label `unsure`. **Skip** stashes the crop to revisit later. The counts table
updates on every action; the current dataset's row is highlighted.

In [4]:
out = widgets.Output()
status = widgets.Label()
counts_box = widgets.HTML()
current = {'sample': None, 'uid': None}


def _refresh_counts():
    render_counts(counts_box, OUT_DIR, MASTER_ZARR_PATHS, CLASSES, DATASET_INDEX)


def _draw():
    sample = current['sample']
    crop = sample['globals'][0, 0].numpy()  # (Z, H, W), t1, channel 0
    img = mip(crop)
    lims = percentile_limits(img, 0.5, 99.5)
    with out:
        clear_output(wait=True)
        fig = Figure(figsize=(5, 5))
        ax = fig.subplots()
        ax.imshow(img, cmap='gray', vmin=lims[0], vmax=lims[1])
        m = sample['meta']
        ax.set_title(f"ds{m['dataset_idx']}  t1={m['t1']}  z={m['z']}", fontsize=10)
        ax.axis('off')
        display(fig)


def _advance():
    while True:
        sample = next(it)
        # a single ZarrCellDataset yields dataset_idx=0; stamp the canonical index
        # so uids/records/counts are scoped to the dataset being annotated.
        sample['meta']['dataset_idx'] = DATASET_INDEX
        uid = make_uid(sample['meta'], int(sample['globals'].shape[-1]))
        if uid not in suppress:
            break
    current['sample'] = sample
    current['uid'] = uid
    status.value = f"dataset [{DATASET_INDEX}]   current: {uid}"
    _draw()


def _make_label_handler(label):
    def handler(_btn):
        rec = save_annotation(OUT_DIR, current['sample'], label,
                              zarr_path=CANONICAL_PATHS[DATASET_INDEX])
        suppress.add(rec['uid'])
        _refresh_counts()
        _advance()
    return handler


def _on_skip(_btn):
    rec = save_skip(OUT_DIR, current['sample'],
                    zarr_path=CANONICAL_PATHS[DATASET_INDEX])
    suppress.add(rec['uid'])
    _refresh_counts()
    _advance()


class_buttons = []
for label in CLASSES:
    b = widgets.Button(description=label, button_style='primary')
    b.on_click(_make_label_handler(label))
    class_buttons.append(b)

unsure_btn = widgets.Button(description='Unsure', button_style='warning')
unsure_btn.on_click(_make_label_handler('unsure'))
skip_btn = widgets.Button(description='Skip', button_style='')
skip_btn.on_click(_on_skip)

controls = widgets.HBox(class_buttons + [unsure_btn, skip_btn])
display(widgets.VBox([status, counts_box, controls, out]))
_refresh_counts()
_advance()

## Review skipped

Walk every crop you skipped and not yet labeled, loading the saved pixels (no
dataset needed). Assign a class to move it into `index.jsonl`, or **Keep
skipped** to leave it. Re-run this cell to refresh the queue after labeling some
— e.g. once you have defined the new class they belong to.

In [ ]:
review_out = widgets.Output()
review_status = widgets.Label()
review_counts_box = widgets.HTML()
rstate = {'queue': load_pending_skips(OUT_DIR), 'i': 0, 'rec': None}


def _review_refresh_counts():
    render_counts(review_counts_box, OUT_DIR, MASTER_ZARR_PATHS, CLASSES, DATASET_INDEX)


def _review_draw():
    rec = rstate['rec']
    crop = np.load(rec['crop_path'])  # (1, Z, H, W)
    img = mip(crop[0])
    lims = percentile_limits(img, 0.5, 99.5)
    with review_out:
        clear_output(wait=True)
        fig = Figure(figsize=(5, 5))
        ax = fig.subplots()
        ax.imshow(img, cmap='gray', vmin=lims[0], vmax=lims[1])
        ax.set_title(rec['uid'], fontsize=9)
        ax.axis('off')
        display(fig)


def _review_advance():
    q = rstate['queue']
    if rstate['i'] >= len(q):
        rstate['rec'] = None
        with review_out:
            clear_output(wait=True)
            print('no more pending skips')
        review_status.value = f'done: reviewed all {len(q)} pending skip(s)'
        return
    rstate['rec'] = q[rstate['i']]
    rstate['i'] += 1
    _review_draw()
    review_status.value = f"pending {rstate['i']}/{len(q)}   uid: {rstate['rec']['uid']}"


def _make_review_handler(label):
    def handler(_btn):
        relabel_skip(OUT_DIR, rstate['rec'], label)
        _review_refresh_counts()
        _review_advance()
    return handler


def _keep_skipped(_btn):
    _review_advance()


review_buttons = []
for label in CLASSES:
    b = widgets.Button(description=label, button_style='primary')
    b.on_click(_make_review_handler(label))
    review_buttons.append(b)
keep_btn = widgets.Button(description='Keep skipped', button_style='')
keep_btn.on_click(_keep_skipped)

review_controls = widgets.HBox(review_buttons + [keep_btn])
display(widgets.VBox([review_status, review_counts_box, review_controls, review_out]))
_review_refresh_counts()
if rstate['queue']:
    _review_advance()
else:
    with review_out:
        print('no pending skips to review')

## Curate / held-out set

Browse the crops already **labeled** for a class and curate them. Pick a class,
choose how many to show **per page** (default 20), and page through with
**Prev**/**Next** — handy for classes with 100+ annotations. Each tile shows the
MIP, its `[dataset_idx] dataset`, and the deterministic `uid` (the unique id you
use to pull the crop again later, e.g. `np.load(rec['crop_path'])`).

Tick the crops you want, then:

- **Add selected to held-out** → appends their records to `heldout.jsonl` (the
  curated set for the linear classifier). Non-destructive; held-out crops stay in
  `index.jsonl` and are marked **★** here.
- **Remove selected (mis-labeled)** → drops them from `index.jsonl` (logged to
  `removed.jsonl`). The pixel files stay on disk, so the live annotator can show
  and re-label them later.

The current page's uids are printed below the grid for easy copy-paste. This
section reads straight from disk, so it does **not** need the dataset/iterator
cells above — but it does reuse the helpers and `OUT_DIR`/`CLASSES`/paths from
the **Config** cell, so run those first.

In [4]:
from collections import Counter

NCOLS = 5  # tiles per row in the gallery
gstate = {'records': [], 'page': 0, 'checks': {}}

g_class = widgets.Dropdown(description='class')
g_perpage = widgets.BoundedIntText(value=20, min=1, max=500, description='per page')
g_prev = widgets.Button(description='◀ Prev')
g_next = widgets.Button(description='Next ▶')
g_pagelbl = widgets.Label()
g_reload = widgets.Button(description='Reload', button_style='')
g_add = widgets.Button(description='Add selected to held-out', button_style='success')
g_remove = widgets.Button(description='Remove selected (mis-labeled)', button_style='danger')
g_status = widgets.Label()
g_counts_box = widgets.HTML()
g_grid = widgets.Output()
g_uidlist = widgets.Output()


def _class_options():
    """(text, value) options for the class dropdown, with per-class counts.

    Always lists CLASSES + 'unsure' (even at count 0) plus any other labels
    actually present, and an 'all' entry first."""
    recs = load_labeled_records(OUT_DIR)
    counts = Counter(r.get('label') for r in recs)
    ordered = list(CLASSES) + ['unsure']
    extras = sorted(l for l in counts if l and l not in ordered)
    opts = [(f'all ({len(recs)})', '__all__')]
    for l in ordered + extras:
        opts.append((f'{l} ({counts.get(l, 0)})', l))
    return opts


def _load_for_class():
    label = None if g_class.value == '__all__' else g_class.value
    gstate['records'] = load_labeled_records(OUT_DIR, label)
    gstate['page'] = 0


def _selected_uids():
    return [uid for uid, chk in gstate['checks'].items() if chk.value]


def _render():
    recs = gstate['records']
    per = max(1, int(g_perpage.value))
    npages = max(1, (len(recs) + per - 1) // per)
    gstate['page'] = min(max(0, gstate['page']), npages - 1)
    start = gstate['page'] * per
    chunk = recs[start:start + per]
    held = load_heldout(OUT_DIR)
    g_pagelbl.value = (
        f"page {gstate['page'] + 1}/{npages} — "
        f"crops {start + 1 if chunk else 0}–{start + len(chunk)} of {len(recs)}"
    )

    gstate['checks'] = {}
    tiles = []
    for rec in chunk:
        uid = rec['uid']
        img_out = widgets.Output()
        with img_out:
            crop = np.load(rec['crop_path'])  # (1, Z, H, W)
            img = mip(crop[0])
            lims = percentile_limits(img, 0.5, 99.5)
            fig = Figure(figsize=(2.4, 2.4))
            ax = fig.subplots()
            ax.imshow(img, cmap='gray', vmin=lims[0], vmax=lims[1])
            ax.axis('off')
            display(fig)
        star = '★ ' if uid in held else ''
        cap = f"{star}[{rec.get('dataset_idx')}] {_ds_short(rec.get('zarr_path') or '')}"
        chk = widgets.Checkbox(value=False, description=uid, indent=False,
                               style={'description_width': 'initial'})
        gstate['checks'][uid] = chk
        tiles.append(widgets.VBox(
            [img_out, widgets.HTML(f"<div style='font-size:11px'>{cap}</div>"), chk],
            layout=widgets.Layout(width='230px', margin='4px'),
        ))

    rows = [widgets.HBox(tiles[i:i + NCOLS]) for i in range(0, len(tiles), NCOLS)]
    with g_grid:
        clear_output(wait=True)
        display(widgets.VBox(rows) if tiles else widgets.Label('no labeled crops for this class'))
    with g_uidlist:
        clear_output(wait=True)
        if chunk:
            print('uids on this page (copy-paste):')
            print('[' + ', '.join(repr(r['uid']) for r in chunk) + ']')
    render_counts(g_counts_box, OUT_DIR, MASTER_ZARR_PATHS, CLASSES, DATASET_INDEX)


def _set_page(p):
    gstate['page'] = p
    _render()


def _refresh_classes(keep_value=True):
    """Rebuild dropdown options after the label set changes; preserve selection."""
    opts = _class_options()
    vals = [v for _, v in opts]
    want = g_class.value if (keep_value and g_class.value in vals) else '__all__'
    g_class.unobserve(_on_class, names='value')
    g_class.options = opts
    g_class.value = want
    g_class.observe(_on_class, names='value')


def _on_class(_change):
    _load_for_class()
    _render()


def _on_add(_btn):
    sel = _selected_uids()
    if not sel:
        g_status.value = 'nothing selected'
        return
    n = export_heldout(OUT_DIR, sel)
    g_status.value = f'added {n} to held-out ({len(sel) - n} already held out)'
    _render()  # refresh ★ markers


def _on_remove(_btn):
    sel = _selected_uids()
    if not sel:
        g_status.value = 'nothing selected'
        return
    n = remove_annotations(OUT_DIR, sel)
    g_status.value = f'removed {n} from index.jsonl (logged to removed.jsonl)'
    _refresh_classes()
    _load_for_class()
    _render()


def _on_reload(_btn):
    _refresh_classes()
    _load_for_class()
    _render()
    g_status.value = 'reloaded'


g_prev.on_click(lambda b: _set_page(gstate['page'] - 1))
g_next.on_click(lambda b: _set_page(gstate['page'] + 1))
g_perpage.observe(lambda c: _render(), names='value')
g_reload.on_click(_on_reload)
g_add.on_click(_on_add)
g_remove.on_click(_on_remove)

controls_row = widgets.HBox([g_class, g_perpage, g_prev, g_next, g_pagelbl, g_reload])
actions_row = widgets.HBox([g_add, g_remove, g_status])
display(widgets.VBox([controls_row, actions_row, g_counts_box, g_grid, g_uidlist]))

g_class.options = _class_options()
g_class.value = '__all__'
g_class.observe(_on_class, names='value')
_load_for_class()
_render()


## Held-out set overview

A read-only montage of everything currently in `heldout.jsonl`, grouped by class
(one labelled row of MIP thumbnails per class). Re-run after curating above to
refresh. Prints a per-class count summary at the top.

In [ ]:
NCOLS_SUMMARY = 8  # thumbnails per row, per class

held = load_heldout_records(OUT_DIR)
if not held:
    print('held-out set is empty — curate crops above first.')
else:
    counts = Counter(r.get('label') for r in held)
    base = list(CLASSES) + ['unsure']
    classes_present = [c for c in base if counts.get(c)] + \
                      sorted(l for l in counts if l and l not in base)

    print(f'held-out total: {len(held)}')
    for c in classes_present:
        print(f'  {c}: {counts[c]}')

    for c in classes_present:
        recs = [r for r in held if r.get('label') == c]
        n = len(recs)
        ncols = min(NCOLS_SUMMARY, n)
        nrows = (n + ncols - 1) // ncols
        fig = Figure(figsize=(1.6 * ncols, 1.7 * nrows))
        axes = fig.subplots(nrows, ncols, squeeze=False)
        for ax in axes.ravel():
            ax.axis('off')
        for i, rec in enumerate(recs):
            ax = axes[i // ncols][i % ncols]
            img = mip(np.load(rec['crop_path'])[0])  # (1, Z, H, W) -> (H, W)
            lo, hi = percentile_limits(img, 0.5, 99.5)
            ax.imshow(img, cmap='gray', vmin=lo, vmax=hi)
            ax.set_title(f"[{rec.get('dataset_idx')}]", fontsize=7)
        fig.suptitle(f'{c}  (n={n})', fontsize=11)
        fig.tight_layout()
        display(fig)
